# Agri-Vision: real-world wheat rust affected-region segmentation

This notebook trains **from random initialization only**. It does not use synthetic data, pretrained encoders, transfer learning, or test data for model selection. The selected task is binary `non-rust` versus `visible stripe-rust affected region`; it is not insurance crop-damage estimation. See `datasets/STRATEGY.md`: NWRD is the primary field-domain dataset; CDTS may be added only after a license/provenance/original-image-group audit.

Validation selects the architecture, loss, and any external-data ablation. The final NWRD original-image groups are frozen and evaluated once.

In [ ]:
!pip -q install tensorflow==2.16.1 pillow pandas scikit-learn matplotlib
import os, json, random, hashlib, time
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from PIL import Image
print('Python/TensorFlow:', os.sys.version, tf.__version__)
SEED=2026; random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Agri-Vision-Damage-Segmentation')
# Existing, read-only source; change only if your Drive path differs.
NWRD=Path('/content/drive/MyDrive/Agri-Vision/NWRD')
for p in [ROOT/'models', ROOT/'results', ROOT/'plots', ROOT/'predictions', ROOT/'configs', ROOT/'logs']:
    p.mkdir(parents=True, exist_ok=True)
assert NWRD.exists(), f'Missing NWRD: {NWRD}'
print('NWRD source:', NWRD)

In [ ]:
# Read-only audit: find paired images/masks, inspect formats, shapes, class IDs, corrupt files, and duplicate candidates.
EXT={'.png','.jpg','.jpeg','.tif','.tiff'}
def files(folder): return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in EXT)
def digest(p): return hashlib.sha256(p.read_bytes()).hexdigest()
splits=[p for p in NWRD.iterdir() if p.is_dir()]
rows=[]; errors=[]
for split in splits:
    ip, mp=split/'images', split/'masks'
    if not ip.exists() or not mp.exists(): continue
    ims={p.stem:p for p in files(ip)}; ms={p.stem:p for p in files(mp)}
    for k in sorted(set(ims)|set(ms)):
        row={'split':split.name,'stem':k,'image':str(ims.get(k,'')),'mask':str(ms.get(k,'')),'paired':k in ims and k in ms}
        try:
            if row['paired']:
                with Image.open(ims[k]) as im: row['image_size']=im.size; row['image_mode']=im.mode
                with Image.open(ms[k]) as m: a=np.asarray(m.convert('L')); row['mask_size']=m.size; row['mask_mode']=m.mode; row['class_ids']=np.unique(a).tolist(); row['pixels']=a.size; row['affected_pixels']=int((a>0).sum())
                row['image_sha256']=digest(ims[k])
        except Exception as e: errors.append({'file':k,'error':str(e)})
        rows.append(row)
audit=pd.DataFrame(rows); audit.to_csv(ROOT/'results'/'nwrd_pair_audit.csv',index=False)
summary={'source':str(NWRD),'pairs':int(audit.paired.sum()),'unpaired':int((~audit.paired).sum()),'corrupt_or_unreadable':errors,'class_ids':sorted({x for ids in audit.class_ids.dropna() for x in ids}),'exact_duplicate_images':audit.image_sha256.dropna().duplicated().sum()}
(ROOT/'results'/'dataset_audit.json').write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2)); display(audit.groupby('split').size())

## Stop and review
Do not train until `dataset_audit.json` confirms the source has valid paired masks and the observed class IDs/semantics have been reviewed. Create a new group-aware 70/15/15 split from original images; do not accept a prior patch-level random split as leakage-safe.

In [ ]:
# REQUIRED INPUT: a reviewed CSV: filename,group_id. Every original image must have a group.
GROUPS=ROOT/'dataset'/'groups.csv'
assert GROUPS.exists(), 'Create groups.csv before splitting.'
g=pd.read_csv(GROUPS); assert {'filename','group_id'} <= set(g.columns)
pairs=audit[audit.paired].copy(); pairs['filename']=pairs.image.map(lambda x:Path(x).name); pairs=pairs.merge(g,on='filename',how='left',validate='many_to_one'); assert pairs.group_id.notna().all()
ids=pairs.group_id.drop_duplicates().sample(frac=1,random_state=SEED).tolist(); n=len(ids); cut1,cut2=round(.70*n),round(.85*n); mapping={x:('train' if i<cut1 else 'validation' if i<cut2 else 'test') for i,x in enumerate(ids)}; pairs['split']=pairs.group_id.map(mapping)
for name in ['train','validation','test']: pairs[pairs.split==name].to_csv(ROOT/'results'/f'{name}.csv',index=False)
print(pairs.groupby('split').size())

In [ ]:
# Select this formulation only after the audit confirms it: 0=non-rust, 1=rust.
# It is a rust affected-region model, not an insurance damage model.
SIZE=(512,512); BATCH=4; CLASSES=2
def read_pair(i,m):
    x=tf.io.decode_image(tf.io.read_file(i),channels=3,expand_animations=False); x.set_shape([None,None,3]); x=tf.cast(tf.image.resize(x,SIZE),tf.float32)/255.
    y=tf.io.decode_image(tf.io.read_file(m),channels=1,expand_animations=False); y.set_shape([None,None,1]); y=tf.cast(tf.squeeze(tf.image.resize(y,SIZE,method='nearest'),-1)>0,tf.int32)
    return x,tf.one_hot(y,CLASSES)
def augment(x,y):
    if tf.random.uniform(())>.5: x,y=tf.image.flip_left_right(x),tf.image.flip_left_right(y)
    return tf.clip_by_value(tf.image.random_brightness(x,.08),0,1),y
def ds(name,training=False):
    d=pd.read_csv(ROOT/'results'/f'{name}.csv'); z=tf.data.Dataset.from_tensor_slices((d.image,d['mask']));
    if training:z=z.shuffle(len(d),seed=SEED)
    z=z.map(read_pair,num_parallel_calls=tf.data.AUTOTUNE); z=z.map(augment,num_parallel_calls=tf.data.AUTOTUNE) if training else z
    return z.batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Standard scratch U-Net. Every convolution uses a fresh random initializer.
def block(x,f):
    for _ in range(2): x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu',kernel_initializer='he_normal')(x)
    return x
def unet():
    inp=tf.keras.Input((*SIZE,3)); x=inp; skips=[]
    for f in [32,64,128,256]: x=block(x,f); skips.append(x); x=tf.keras.layers.MaxPool2D()(x)
    x=block(x,512)
    for f,s in zip([256,128,64,32],reversed(skips)): x=tf.keras.layers.Conv2DTranspose(f,2,2,padding='same')(x); x=block(tf.keras.layers.Concatenate()([x,s]),f)
    return tf.keras.Model(inp,tf.keras.layers.Conv2D(CLASSES,1,activation='softmax')(x))
def dice_loss(y,p):
    n=2*tf.reduce_sum(y*p,axis=(0,1,2))+1; d=tf.reduce_sum(y+p,axis=(0,1,2))+1; return 1-tf.reduce_mean(n/d)
model=unet(); model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss=lambda y,p:tf.keras.losses.categorical_crossentropy(y,p)+dice_loss(y,p),metrics=['accuracy']); model.summary()

In [ ]:
# Select architecture/loss from validation only. Leave test untouched until one final run.
callbacks=[tf.keras.callbacks.ModelCheckpoint(ROOT/'models'/'candidate_best.keras',save_best_only=True,monitor='val_loss'),tf.keras.callbacks.EarlyStopping(patience=15,restore_best_weights=True),tf.keras.callbacks.ReduceLROnPlateau(patience=6)]
history=model.fit(ds('train',True),validation_data=ds('validation'),epochs=100,callbacks=callbacks)
pd.DataFrame(history.history).to_csv(ROOT/'results'/'training_history.csv',index=False)
model.save(ROOT/'models'/'candidate_final.keras')
(ROOT/'configs'/'training_config.json').write_text(json.dumps({'seed':SEED,'input_size':SIZE,'batch_size':BATCH,'architecture':'scratch_unet','weights':'random_initialization','task':'binary_rust_vs_non_rust'},indent=2))

In [ ]:
# VALIDATION-ONLY experiment comparison: record each candidate before the final test is unlocked.
# Change architecture/loss/input size here, train again, and enter validation metrics only.
EXPERIMENTS=ROOT/'results'/'experiment_results.csv'
pd.DataFrame(columns=['experiment','architecture','loss','input_size','parameters','batch_size','learning_rate','epochs','training_seconds','validation_pixel_accuracy','validation_precision_macro','validation_recall_macro','validation_f1_macro','validation_mean_iou','validation_mean_dice','affected_iou','affected_dice']).to_csv(EXPERIMENTS,index=False)
print('Do not add final-test metrics to this comparison table.')

In [ ]:
# FINAL TEST: run once, only after model/loss/architecture are fixed by validation.
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
FINAL=ROOT/'results'/'final_test'; FINAL.mkdir(parents=True,exist_ok=True); (FINAL/'prediction_visualizations').mkdir(exist_ok=True)
selected=tf.keras.models.load_model(ROOT/'models'/'candidate_best.keras',compile=False)
truths=[]; predictions=[]; areas=[]; examples=[]
for images, onehot in ds('test'):
    probs=selected.predict(images,verbose=0); pred=np.argmax(probs,-1); truth=np.argmax(onehot.numpy(),-1)
    for x,yhat,y in zip(images.numpy(),pred,truth):
        truths.extend(y.ravel()); predictions.extend(yhat.ravel());
        gt=100*(y==1).mean(); pr=100*(yhat==1).mean(); areas.append((gt,pr)); examples.append((x,y,yhat))
cm=confusion_matrix(truths,predictions,labels=[0,1]); p,r,f,s=precision_recall_fscore_support(truths,predictions,labels=[0,1],zero_division=0)
iou=np.diag(cm)/(cm.sum(0)+cm.sum(1)-np.diag(cm)); dice=2*np.diag(cm)/(2*np.diag(cm)+(cm.sum(0)-np.diag(cm))+(cm.sum(1)-np.diag(cm)))
metrics={'evaluation_split':'frozen_unseen_test','pixel_accuracy':float(np.trace(cm)/cm.sum()),'precision_macro':float(p.mean()),'recall_macro':float(r.mean()),'f1_macro':float(f.mean()),'mean_iou':float(iou.mean()),'mean_dice':float(dice.mean()),'classes':{'non_rust':{'precision':float(p[0]),'recall':float(r[0]),'f1':float(f[0]),'iou':float(iou[0]),'dice':float(dice[0])},'rust_affected_region':{'precision':float(p[1]),'recall':float(r[1]),'f1':float(f[1]),'iou':float(iou[1]),'dice':float(dice[1])}},'affected_area_mae_percentage_points':float(np.mean(np.abs(np.array(areas)[:,0]-np.array(areas)[:,1])))}
(FINAL/'final_metrics.json').write_text(json.dumps(metrics,indent=2)); pd.json_normalize(metrics,sep='_').to_csv(FINAL/'final_metrics.csv',index=False)
fig,ax=plt.subplots(figsize=(5,4)); ax.imshow(cm,cmap='Blues'); ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['non-rust','rust'],yticklabels=['non-rust','rust'],xlabel='Prediction',ylabel='Ground truth'); [ax.text(j,i,str(cm[i,j]),ha='center',va='center') for i in range(2) for j in range(2)]; fig.tight_layout(); fig.savefig(FINAL/'confusion_matrix.png',dpi=160); plt.show(); print(json.dumps(metrics,indent=2))
# Deterministic, spread-out samples rather than only easy examples.
for n in np.linspace(0,len(examples)-1,min(12,len(examples)),dtype=int):
    x,y,yp=examples[n]; fig,axs=plt.subplots(1,3,figsize=(12,4)); axs[0].imshow(x); axs[0].set_title('Image'); axs[1].imshow(y,cmap='magma',vmin=0,vmax=1); axs[1].set_title('Ground truth'); axs[2].imshow(yp,cmap='magma',vmin=0,vmax=1); axs[2].set_title('Prediction'); [a.axis('off') for a in axs]; fig.tight_layout(); fig.savefig(FINAL/'prediction_visualizations'/f'test_{n:04d}.png',dpi=150); plt.close(fig)
selected.save(ROOT/'models'/'best_affected_region_segmentation.keras')
print('Saved final model/results to',ROOT)

## Interpretation guardrail
The `affected_area_mae_percentage_points` measures predicted **rust-region area as a share of image pixels**. It is not crop damage percentage and must not be used for insurance decisions without masks distinguishing healthy crop, damaged crop, and background.